# Stage 3 — Build Index (Chroma + SQLite)

Builds the persistent vector index from the Stage 2 outputs:

```
data/redacted/text/AG####.txt   (redacted offers)
data/extracted/AG####.json      (extractions: angebot_id, datum, preis)
        │
        ├──► Chunking (SentenceSplitter 1500/300)
        │        │
        │        ▼
        │   Embedding (nomic-embed-text via Ollama)
        │        │
        │        ▼
        └──► ChromaDB  data/db/chroma/   collection "offers"
             SQLite    data/db/sql/offers.db   offers(angebot_id, datum, preis)
```

**Chunk metadata** (the "world facts" — exactly the schema the app's
`retriever.py` reads):

| Key | Type | Meaning |
|---|---|---|
| `angebot_id` | str | Join key (AG####), used for metadata filters + citations |
| `datum` | str or null | Offer date, ISO (JJJJ-MM-TT) |
| `preis` | float or null | Net total in EUR, null when no explicit total |
| `abschnitt` | str | Always `"volltext"` (whole offer, not a section) |

The collection is **dropped and rebuilt on every run** → repeatable.
SQLite is the structured counterpart (aggregation, presentation) — the app
does not query it.

**Next:**
- `04-retrieval-demo-rag.ipynb` (Stage 3b) — pure RAG: hybrid BM25 + vector +
  RRF → LLM rerank → refusal gate → cited answer with verbatim quotes and
  page numbers (`[AG#### | S. X]`).
- `05-retrieval-demo-full.ipynb` (Stage 3c) — the app's router on top of the
  same retrieval core: deterministic price answers, clarification,
  aggregation limits, ID-aware retrieval.

## Setup — Environment, Paths & Embedding Model

Load `.env`, define `CHROMA_DIR` / `SQLITE_DIR`, create the Ollama embedding client (nomic-embed-text).

In [1]:
import os, json, shutil
from pathlib import Path
from dotenv import load_dotenv

def rel(p):
    """Render a path relative to the repo root (or ~) for display."""
    root = Path.cwd().parent
    p = Path(p)
    try:
        return str(p.relative_to(root))
    except ValueError:
        try:
            return "~/" + str(p.relative_to(Path.home()))
        except ValueError:
            return str(p)


# Load env files (walk up: notebooks/ -> repo root). .env.example provides
# the defaults, .env overrides them (same layering as the app's config.py).
for path in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    example, env = path / ".env.example", path / ".env"
    if example.exists():
        load_dotenv(example, override=False)
        print(f"✅ Loaded defaults from {rel(example)}")
    if env.exists():
        load_dotenv(env, override=True)
        print(f"✅ Loaded .env from {rel(env)}")
    if example.exists() or env.exists():
        break

# Secrets layer (outside the repo, chmod 600): highest priority, keeps API
# keys out of the workspace (same as the app's config.py).
_secrets = Path.home() / ".config" / "rag-quote-history" / "secrets.env"
if _secrets.exists():
    load_dotenv(_secrets, override=True)
    print(f"\u2705 Loaded secrets from {rel(_secrets)}")

EMBED_BASE_URL = os.getenv("EMBED_BASE_URL", "")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")

DEMO_DIR = Path.cwd().parent            # 01-submission
REDACTED_TEXT_DIR = DEMO_DIR / "data" / "redacted" / "text"
EXTRACTED_DIR = DEMO_DIR / "data" / "extracted"
CHROMA_DIR = DEMO_DIR / "data" / "db" / "chroma"
SQLITE_DIR = DEMO_DIR / "data" / "db" / "sql"
SQLITE_PATH = SQLITE_DIR / "offers.db"
for d in (CHROMA_DIR, SQLITE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Chunking (from .env / .env.example)
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "1500"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "300"))

from llama_index.embeddings.ollama import OllamaEmbedding
embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=EMBED_BASE_URL.replace("/v1", ""),
)
print(f"✅ Embeddings: {EMBED_MODEL} at {EMBED_BASE_URL}")
print(f"📁 Chroma dir: {rel(CHROMA_DIR)}")
print(f"📁 SQLite: {rel(SQLITE_PATH)}")

✅ Loaded defaults from .env.example
✅ Loaded .env from .env
✅ Loaded secrets from ~/.config/rag-quote-history/secrets.env


✅ Embeddings: nomic-embed-text at <URL>
📁 Chroma dir: data/db/chroma
📁 SQLite: data/db/sql/offers.db


In [2]:
# Set CLEAN_SLATE = True to wipe the index artifacts and rebuild from scratch.
# Default False: the Chroma collection is dropped & rebuilt in the next cell
# anyway, so re-running is safe either way. Stage 2 outputs are NEVER touched.
CLEAN_SLATE = True

if CLEAN_SLATE:
    for d in (CHROMA_DIR, SQLITE_DIR):
        if d.exists():
            shutil.rmtree(d)
            print(f"Removed {rel(d)}")
        d.mkdir(parents=True, exist_ok=True)
    print("Clean slate — index artifacts will be rebuilt")
else:
    print("Normal mode — collection will be dropped & rebuilt in the next cell")

Removed data/db/chroma
Removed data/db/sql
Clean slate — index artifacts will be rebuilt


## Step 1: Load Stage 2 outputs

All redacted texts + extraction JSONs, joined on `angebot_id`.
The count is derived from the files on disk (no hard-coded number —
the submission repo runs the same notebook with fewer offers).

In [3]:
texts = {}
for f in sorted(REDACTED_TEXT_DIR.glob("AG*.txt")):
    texts[f.stem] = f.read_text()

extractions = {}
for f in sorted(EXTRACTED_DIR.glob("AG*.json")):
    extractions[f.stem] = json.loads(f.read_text())

n = len(texts)
assert n == len(extractions), f"text/extracted mismatch: {len(texts)} vs {len(extractions)}"
assert n > 0, "no offers found — check data/redacted/text/ and data/extracted/"
missing = set(texts) - set(extractions)
assert not missing, f"offers without extraction: {sorted(missing)}"

n_with_price = sum(1 for d in extractions.values() if d.get("preis") is not None)
print(f"✅ Loaded {n} offers ({n_with_price} with explicit price)")
print()
print(f"{'angebot_id':<12} {'datum':<12} {'preis':>10}  chars")
for oid in sorted(texts)[:5]:
    d = extractions[oid]
    preis = d.get("preis")
    print(f"{oid:<12} {str(d.get('datum')):<12} {str(preis):>10}  {len(texts[oid]):>6}")
print("...")

✅ Loaded 10 offers (9 with explicit price)

angebot_id   datum             preis  chars
AG1001       2024-03-12         7380     970
AG1002       2024-07-08         5650     982
AG1003       2025-01-14        19300    1067
AG1004       2025-04-02         2320     838
AG1005       2025-05-03         None    1091
...


## Step 2: Build Chroma Index (chunking + embedding)

Drop & rebuild the `offers` collection. One `Document` per offer →
`SentenceSplitter(1500 tokens, 300 overlap)` → embed → store with metadata
`{angebot_id, abschnitt, datum?, preis?}`.

In [4]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext, Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore

COLLECTION = "offers"

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    chroma_client.delete_collection(COLLECTION)
    print(f"🗑️  Dropped existing '{COLLECTION}' collection")
except Exception:
    pass
chroma_collection = chroma_client.get_or_create_collection(COLLECTION)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# One Document per offer; metadata = the "world facts" only.
# Chroma rejects None metadata values -> omit the key when null
# (the app reads missing keys as None via dict.get()).
documents = []
for offer_id, text in texts.items():
    d = extractions[offer_id]
    meta = {"angebot_id": offer_id, "abschnitt": "volltext"}
    if d.get("datum") is not None:
        meta["datum"] = d["datum"]
    if d.get("preis") is not None:
        meta["preis"] = float(d["preis"])
    documents.append(Document(text=text, metadata=meta))

splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model,
    transformations=[splitter],
    storage_context=storage_context,
    show_progress=True,
)

n_chunks = chroma_collection.count()
print(f"✅ Chroma index '{COLLECTION}': {len(documents)} offers -> {n_chunks} chunks in {rel(CHROMA_DIR)}/")

# Per-offer chunk distribution
from collections import Counter
dist = Counter()
for m in chroma_collection.get(include=["metadatas"])["metadatas"]:
    dist[m["angebot_id"]] += 1
buckets = Counter(dist.values())
print("Chunks per offer:", {f"{k} chunk(s)": f"{v} offers" for k, v in sorted(buckets.items())})

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Chroma index 'offers': 10 offers -> 10 chunks in data/db/chroma/
Chunks per offer: {'1 chunk(s)': '10 offers'}


## Step 3: SQLite (structured metadata) + Smoke Check

Create `offers(angebot_id PK, datum, preis)` from the extracted JSONs.
Then one quick vector query to prove the index works end-to-end.

In [5]:
import sqlite3

conn = sqlite3.connect(SQLITE_PATH)
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS offers")
cur.execute("""
CREATE TABLE offers (
    angebot_id TEXT PRIMARY KEY,
    datum      TEXT,
    preis      REAL
)
""")
for offer_id, d in extractions.items():
    cur.execute("INSERT INTO offers VALUES (?,?,?)",
                (offer_id, d.get("datum"), d.get("preis")))
conn.commit()

rows = cur.execute("SELECT COUNT(*), SUM(preis IS NOT NULL) FROM offers").fetchone()
print(f"✅ SQLite: {rows[0]} rows, {rows[1]} with price in {rel(SQLITE_PATH)}")
print()
print("Sample rows:")
for r in cur.execute("SELECT * FROM offers ORDER BY angebot_id LIMIT 5"):
    print("  ", r)
conn.close()

# --- Smoke check: one vector query against the fresh index ---
print()
print("Smoke check — vector query 'Zahlungsbedingungen und Skonto':")
query = "Zahlungsbedingungen und Skonto"
q_emb = embed_model.get_text_embedding(query)
res = chroma_collection.query(query_embeddings=[q_emb], n_results=3,
                              include=["metadatas", "distances"])
for meta, dist in zip(res["metadatas"][0], res["distances"][0]):
    score = max(0.0, 1.0 - dist / 2.0)   # L2 distance -> similarity, like the app
    print(f"  {meta['angebot_id']} | {meta.get('datum', '—')} | "
          f"preis={meta.get('preis', '—')} | score={score:.3f}")

✅ SQLite: 10 rows, 9 with price in data/db/sql/offers.db

Sample rows:
   ('AG1001', '2024-03-12', 7380.0)
   ('AG1002', '2024-07-08', 5650.0)
   ('AG1003', '2025-01-14', 19300.0)
   ('AG1004', '2025-04-02', 2320.0)
   ('AG1005', '2025-05-03', None)

Smoke check — vector query 'Zahlungsbedingungen und Skonto':
  AG1001 | 2024-03-12 | preis=7380.0 | score=0.665
  AG1006 | 2025-09-21 | preis=8300.0 | score=0.664
  AG1004 | 2025-04-02 | preis=2320.0 | score=0.660
